## LIBRARIES AND DECLARATION

In [1]:
import os
import json
import time
import pandas as pd
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
from IPython.display import FileLink

## CONFIG PROVIDER

In [2]:
secrets = UserSecretsClient()
NINE_ROUTER_API_KEY = secrets.get_secret("NINE_ROUTER_API_KEY")

MODEL_CONFIG = {
    "base_url": "https://r8slvik.abc-tunnel.us/v1",
    "model": "cx/gpt-5.6-sol",
    "api_key": NINE_ROUTER_API_KEY
}

JUDGE_MODEL = MODEL_CONFIG["model"]

NOISE_FILE = '/kaggle/input/datasets/dhuyent/classified-filtered-stage2/step1_filter2_reference1.csv'
CHECKPOINT_FILE = 'step2_filter2_reference1_gpt-5.6-sol_2.csv'

MAX_RETRIES = 3
ROW_PAUSE   = 2

client = OpenAI(
    base_url=MODEL_CONFIG["base_url"],
    api_key=MODEL_CONFIG["api_key"]
)

# PROMPT_STRATEGY = "zero-shot" # or "few-shot"
PROMPT_STRATEGY = "few-shot"

## HELPERS

In [3]:
def call_llm(prompt, **kwargs):
    params = {
        "model": MODEL_CONFIG["model"],
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    }
    params.update(kwargs)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(**params)
            content = response.choices[0].message.content
            if not content:
                raise ValueError("Model returned empty response")
            return content.strip()
        except Exception as e:
            print(f"  API error (attempt {attempt}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES:
                print("  Retrying in 20s...")
                time.sleep(20)

    return None

In [4]:
def evaluate_with_judge(problem, code):
    # # Zero-shot
    # judge_prompt = f"""
    # ### You are an expert code reviewer.
    # ### You will be given a problem and a submitted solution.
    # `````Problem {problem} ```
    # ````Submission {code} ```
    # ### Your task is to evaluate the submission.
    # ### Assign exactly one of the following verdicts:
    # - Accepted: Code is logically correct and satisfies ALL constraints. This is the ONLY verdict that counts as correct.
    # - Wrong Answer: Code produces incorrect output for some input.
    # - Time Limit Exceeded: Correct output BUT time complexity exceeds the stated bound.
    # - Memory Limit Exceeded: Correct output BUT space complexity exceeds the stated bound.
    # - Runtime Error: Code causes a crash at runtime (e.g. division by zero, index out of bounds).
    # - Compilation Error: Code has syntax or fails to compile before execution.
    # - Unknown: Verdict cannot be determined.
    # ### You should respond in the format of a JSON object:
    # {{
    #   "pred_status": "<Accepted | Wrong Answer | Time Limit Exceeded | Memory Limit Exceeded | Runtime Error | Compilation Error | Unknown>",
    #   "pred_failed_test_case": {{
    #   "pred_input": "...",
    #   "pred_expected_output": "...", // null if program did not produce output 
    #   "pred_actual_output": "..." // null if program did not terminate
    #   }},
    #   "pred_reason": "..."
    # }}
    # Note: Include "pred_failed_test_case" for: Wrong Answer, Runtime Error, Time Limit Exceeded, Memory Limit Exceeded — whenever a specific (or last executed) input can be identified. Include "pred_reason" for: TLE, MLE, Runtime Error, Compilation Error, Unknown. Omit both fields for Accepted. For TLE/MLE, set "pred_expected_output" and "pred_actual_output" to null since the program did not finish. Return only valid JSON. No extra text.
    # """

    # Few-shot
    judge_prompt = f"""
    ### You are an expert code reviewer.
    ### You will be given a problem and a submitted solution.
    `````Problem {problem} ```
    ````Submission {code} ```
    ### Your task is to evaluate the submission.
    ### Assign exactly one of the following verdicts:
    - Accepted: Code is logically correct and satisfies ALL constraints. This is the ONLY verdict that counts as correct.
    - Wrong Answer: Code produces incorrect output for some input.
    - Time Limit Exceeded: Correct output BUT time complexity exceeds the stated bound.
    - Memory Limit Exceeded: Correct output BUT space complexity exceeds the stated bound.
    - Runtime Error: Code causes a crash at runtime (e.g. division by zero, index out of bounds).
    - Compilation Error: Code has syntax or fails to compile before execution.
    - Unknown: Verdict cannot be determined.
    ### You should respond in the format of a JSON object:
    {{
     "pred_status": "<Accepted | Wrong Answer | Time Limit Exceeded | Memory Limit Exceeded | Runtime Error | Compilation Error | Unknown>",
     "pred_failed_test_case": {{
     "pred_input": "...",
     "pred_expected_output": "...", // null if program did not produce output 
     "pred_actual_output": "..." // null if program did not terminate
     }},
     "pred_reason": "<1-sentence explanation>"
    }}
    Note: Include "pred_failed_test_case" for: Wrong Answer, Runtime Error, Time Limit Exceeded, Memory Limit Exceeded - whenever a specific (or last executed) input can be identified. Include "pred_reason" for: TLE, MLE, Runtime Error, Compilation Error, Unknown. Omit both fields for Accepted. For TLE/MLE, set "pred_expected_output" and "pred_actual_output" to null since the program did not finish. Return only valid JSON. No extra text.

    ### Examples:
    ## 1 - Accepted
    Problem: Given an array of integers, return the sum of all elements. Constraints: Time O(n)
    Submission:
    int sum(vector<int>& nums) {{
        int total = 0;
        for (int x : nums) total += x;
        return total;
    }}
    Response: {{"pred_status":"Accepted"}}
    
    ## 2 - Wrong Answer
    Problem: Given an array of integers, return the sum of all elements.
    Submission:
    int sum(vector<int>& nums) {{
        int total = 0;
        for (int i = 0; i < nums.size() - 1; i++) total += nums[i];
        return total;
    }}
    Response: {{"pred_status":"Wrong Answer","pred_failed_test_case":{{"pred_input":"nums = [1, 2, 3]","pred_expected_output":"6","pred_actual_output":"3"}}}}
    
    ## 3 - Time Limit Exceeded
    Problem: Given an array of n integers, find if any two elements sum to 0. Constraints: Time O(n log n), 1<=n<=10^6
    Submission:
    bool hasPairSumZero(vector<int>& nums) {{
        for (int i = 0; i < nums.size(); i++)
            for (int j = i+1; j < nums.size(); j++)
                if (nums[i] + nums[j] == 0) return true;
        return false;
    }}
    Response: {{"pred_status":"Time Limit Exceeded","pred_reason":"Nested loops give O(n²) time complexity, exceeding the required O(n log n) bound.","pred_failed_test_case":{{"pred_input":"nums = [1, 2, 3, ..., n=10^6]","pred_expected_output":null,"pred_actual_output":null}}}}
    
    ## 4 - Memory Limit Exceeded
    Problem: Given an array of n integers and a target, determine if any two elements sum to target. Constraints: Time O(n), Space O(1), 1<=n<=10^7
    Submission:
    bool hasTargetSum(vector<int>& nums, int target) {{
        vector<vector<bool>> seen(nums.size(), vector<bool>(nums.size(), false));
        for (int i = 0; i < nums.size(); i++)
            for (int j = 0; j < nums.size(); j++)
                if (!seen[i][j] && nums[i] + nums[j] == target) return true;
        return false;
    }}
    Response: {{"pred_status":"Memory Limit Exceeded","pred_reason":"Allocates an O(n²) boolean matrix, exceeding the required O(1) space bound.","pred_failed_test_case":{{"pred_input":"nums.size() = 10^7, target = 0","pred_expected_output":null,"pred_actual_output":null}}}}
    
    ## 5 - Runtime Error
    Problem: Return the first element of an array. Array is guaranteed non-empty.
    Submission:
    int first(vector<int>& nums) {{
        return nums.at(0);
    }}
    Response: {{"pred_status":"Runtime Error","pred_reason":"The array is empty, so accessing index 0 with .at() throws an out-of-range exception.","pred_failed_test_case":{{"pred_input":"nums = []","pred_expected_output":"undefined (invalid input per constraints)","pred_actual_output":"std::out_of_range exception"}}}}
    
    ## 6 - Compilation Error
    Problem: Return the maximum of two integers.
    Submission:
    int maxTwo(int a, int b) {{
        return a > b ? a : b
    }}
    Response: {{"pred_status":"Compilation Error","pred_reason":"Missing semicolon after the return statement."}}
    
    ## 7 - Unknown
    Problem: (empty)
    Submission:
    int solve(vector<int>& nums) {{
        return nums.size();
    }}
    Response: {{"pred_status":"Unknown","pred_reason":"The verdict cannot be determined because the problem requirements are incomplete."}}
    """

    raw = call_llm(judge_prompt, max_tokens=1024, temperature=0.0)
    if not raw:
        return None

    try:
        # Strip markdown fences if present
        clean = raw.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        return json.loads(clean.strip())
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")
        print(f"  Raw response: {raw[:300]}")
        return None

## LOAD CHECKPOINT

In [5]:
if os.path.exists(CHECKPOINT_FILE):
    done_df = pd.read_csv(CHECKPOINT_FILE)
    # Composite key: (id, llm_model, prompt_strategy)
    done_ids = set(
        zip(done_df['id'], done_df['llm_model'], done_df['prompt_strategy'])
    )
    final_results = done_df.to_dict('records')
    print(f"Resumed: {len(done_ids)} records already processed, continuing from where we left off...")
else:
    done_ids = set()
    final_results = []
    print("Starting fresh...")

Starting fresh...


## PROCESS DATA

In [6]:
def normalize_model_name(model_id: str) -> str:
    """Bỏ các prefix vendor khi lưu vào CSV."""
    for prefix in ("openai/", "qwen/", "groq/", "cx/"):
        if model_id.startswith(prefix):
            return model_id[len(prefix):]
    return model_id

In [7]:
print("Reading intermediate dataset (with noise)...")
df_noise = pd.read_csv(NOISE_FILE)
print(f"Total records to process: {len(df_noise)}")

for index, row in df_noise.iterrows():
    row_llm_model       = row.get('llm_model', JUDGE_MODEL)
    row_prompt_strategy = row.get('prompt_strategy', PROMPT_STRATEGY)
    composite_key = (row['id'], row_llm_model, row_prompt_strategy)

    if composite_key in done_ids:
        print(f"Skipping ID {row['id']} | {row_llm_model} | {row_prompt_strategy} (already done)")
        continue

    print(f"Processing ID {row['id']} | {row_llm_model} | {row_prompt_strategy} - judging...")

    judge_data = evaluate_with_judge(row['problem'], row['buggy_submission'])

    if judge_data:
        failed_test = judge_data.get('pred_failed_test_case') or {}
        row_dict = row.to_dict()
        row_dict['llm_model']            = normalize_model_name(row_llm_model)
        row_dict['prompt_strategy']      = row_prompt_strategy
        row_dict['pred_status']          = judge_data.get('pred_status')
        row_dict['pred_input']           = json.dumps(failed_test.get('pred_input', ""))
        row_dict['pred_actual_output']   = json.dumps(failed_test.get('pred_actual_output', ""))
        row_dict['pred_expected_output'] = json.dumps(failed_test.get('pred_expected_output', ""))
        row_dict['pred_reason']          = judge_data.get('pred_reason', "")

        final_results.append(row_dict)
        done_ids.add(composite_key)

        pd.DataFrame(final_results).to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8-sig')
        print(f"  Checkpoint saved ({len(final_results)} records)")
    else:
        print(f"  Skipping ID {row['id']} — judge returned None")

    time.sleep(ROW_PAUSE)

print(f"\nDone! Saved {len(final_results)} records to {CHECKPOINT_FILE}")

Reading intermediate dataset (with noise)...
Total records to process: 35
Processing ID 18 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (1 records)
Processing ID 20 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (2 records)
Processing ID 29 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (3 records)
Processing ID 35 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (4 records)
Processing ID 46 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (5 records)
Processing ID 53 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (6 records)
Processing ID 57 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (7 records)
Processing ID 60 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (8 records)
Processing ID 61 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (9 records)
Processing ID 63 | cx/gpt-5.6-sol | few-shot - judging...
  Checkpoint saved (10 records)
Processing ID 71 | cx/gpt-5.6-sol |

## CHECK MISSING IDs

In [8]:
# So sánh composite key (id, llm_model, prompt_strategy) vì cùng một id có thể được judge bởi nhiều model / prompt strategy khác nhau
print("Checking missing IDs...")
df_noise = pd.read_csv(NOISE_FILE)
done_df  = pd.read_csv(CHECKPOINT_FILE)

all_ids     = set(df_noise['id'].astype(str))
done_ids    = set(done_df['id'].astype(str))
missing_ids = all_ids - done_ids

if not missing_ids:
    print("No missing IDs, dataset is complete!")
else:
    print(f"Found {len(missing_ids)} missing IDs: {sorted(missing_ids)}")
    missing_df = df_noise[
        df_noise['id'].astype(str).isin(missing_ids)
    ].reset_index(drop=True)

    final_results = done_df.to_dict('records')
    for _, row in missing_df.iterrows():
        row_llm_model       = row.get('llm_model', JUDGE_MODEL)
        row_prompt_strategy = row.get('prompt_strategy', PROMPT_STRATEGY)

        print(f"Retrying missing ID {row['id']} | {row_llm_model} | {row_prompt_strategy}...")
        judge_data = evaluate_with_judge(row['problem'], row['buggy_submission'])
        if judge_data:
            failed_test = judge_data.get('pred_failed_test_case') or {}
            row_dict = row.to_dict()
            row_dict['llm_model']            = normalize_model_name(row_llm_model)
            row_dict['prompt_strategy']      = row_prompt_strategy
            row_dict['pred_status']          = judge_data.get('pred_status')
            row_dict['pred_input']           = json.dumps(failed_test.get('pred_input', ""))
            row_dict['pred_actual_output']   = json.dumps(failed_test.get('pred_actual_output', ""))
            row_dict['pred_expected_output'] = json.dumps(failed_test.get('pred_expected_output', ""))
            row_dict['pred_reason']          = judge_data.get('pred_reason', "")
            final_results.append(row_dict)
            results_df = pd.DataFrame(final_results).sort_values('id').reset_index(drop=True)
            results_df.to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8-sig')
            print(f"  Checkpoint saved ({len(final_results)} records)")
        else:
            print(f"  Failed to judge ID {row['id']}, skipping...")
        time.sleep(ROW_PAUSE)

    print(f"\nDone! Saved {len(final_results)} records to {CHECKPOINT_FILE}")

Checking missing IDs...
No missing IDs, dataset is complete!


## SAVE OUTPUT

In [9]:
FileLink(r'step2_filter2_reference1_gpt-5.6-sol_2.csv')

/kaggle/working/step2_filter2_reference1_gpt-5.6-sol_2.csv